# AgroFácil · Pipeline Bronze → Prata → Ouro (Databricks)

Notebook único, executado de cima para baixo no schema `workspace.gs` (Volume `/Volumes/workspace/gs/chegada`).
Os arquivos de `chegada/` são gerados pelo DAG do Airflow (`dags/agrofacil_coleta_dados.py`) e enviados ao Volume em dois lotes.

| Requisito | Onde está neste notebook |
| :-- | :-- |
| 04 · Lotes no Volume | seção **00** (listagem do Volume depois de cada lote) |
| 05 · Bronze incremental | seção **01** (Auto Loader, contagens nos 3 momentos, `DESCRIBE HISTORY`) |
| 06 · Prata | seção **02** (tipagem, chaves, regras de qualidade, `MERGE` duas vezes) |
| 07 · Ouro e consultas SQL | seções **03** (star schema, integridade, view do MVP) e **04** (5 consultas com interpretação) |
| 08 · LGPD e MVP | seções **05** (hash com sal e máscara) e **06** (extrato `dados_ouro.json`) |

Os requisitos 01 a 03 (fontes, DAG, paralelismo e credenciais) estão no DAG e no `README.md` do repositório.

# AgroFácil · 00 · Setup do Volume, reset e listagem dos lotes

Esta seção (a) cria o schema e o Volume, (b) opcionalmente **zera tudo** para refazer a demonstração do incremental
e (c) registra a **listagem do Volume depois de cada lote** (requisito 4).

### Roteiro da demonstração (requisitos 4 e 5)
1. Para refazer do zero, e **somente antes de subir arquivos**, rode o reset com `resetar = sim` e `apagar_arquivos = sim` (zera tabelas, checkpoints e os JSON antigos do Volume). Depois de subir os lotes, mantenha `apagar_arquivos = nao`.
2. Suba o **lote 1** (primeira metade dos dias) nas pastas `clima/`, `ibge_municipios/` e `lavouras/`.
3. Com `momento = 1_apos_lote1`, rode a listagem do Volume e depois a seção 01 (Bronze).
4. Suba o **lote 2** (segunda metade dos dias) e repita com `momento = 2_apos_lote2`.
5. Rode a seção 01 de novo **sem arquivo novo**, com `momento = 3_reexecucao_sem_arquivo_novo`.

Depois do 3º momento, rode as seções 02 (Prata), 03 (Ouro), 04 (consultas), 05 (LGPD) e 06 (extrato do MVP), nesta ordem.

In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/gs/chegada"
PASTAS = ["clima", "ibge_municipios", "lavouras"]
TABELAS = [
    "bronze_clima", "bronze_ibge_municipios", "bronze_lavouras",
    "prata_clima", "prata_ibge_municipios", "prata_lavouras", "prata_qualidade_regras",
    "ouro_fato_clima_lavoura", "ouro_dim_tempo", "ouro_dim_local",
    "governanca_lgpd_produtores", "log_contagens_bronze", "log_listagem_volume",
]

dbutils.widgets.dropdown("resetar", "nao", ["nao", "sim"], "Apagar tabelas e checkpoints?")
dbutils.widgets.dropdown("apagar_arquivos", "nao", ["nao", "sim"], "Apagar os JSON das pastas do Volume?")
dbutils.widgets.text("momento", "1_apos_lote1", "Momento da listagem")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gs")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.gs.chegada")
for p in PASTAS + ["_controle"]:
    dbutils.fs.mkdirs(f"{BASE}/{p}")
print("Schema e Volume prontos:", BASE)

Schema e Volume prontos: /Volumes/workspace/gs/chegada


In [0]:
if dbutils.widgets.get("resetar") == "sim":
    spark.sql("DROP VIEW IF EXISTS workspace.gs.ouro_mvp_diario")
    for t in TABELAS:
        spark.sql(f"DROP TABLE IF EXISTS workspace.gs.{t}")
    dbutils.fs.rm(f"{BASE}/_controle", True)
    dbutils.fs.mkdirs(f"{BASE}/_controle")
    print("Reset concluído: tabelas, view e checkpoints apagados.")
else:
    print("Reset não solicitado (resetar = nao).")

if dbutils.widgets.get("apagar_arquivos") == "sim":
    for p in PASTAS:
        try:
            for f in dbutils.fs.ls(f"{BASE}/{p}"):
                if f.name.endswith(".json"):
                    dbutils.fs.rm(f.path)
            print(f"JSON removidos de {p}/")
        except Exception:
            print(f"{p}/ não existe (ok)")
else:
    print("Arquivos do Volume mantidos (apagar_arquivos = nao).")

Reset não solicitado (resetar = nao).
Arquivos do Volume mantidos (apagar_arquivos = nao).


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gs.log_listagem_volume (
  momento STRING, pasta STRING, qtd_arquivos INT, primeiro_arquivo STRING, ultimo_arquivo STRING,
  registrado_em TIMESTAMP
) USING DELTA
""")


def listar_volume(momento):
    """Lista o Volume por pasta, imprime arquivo a arquivo e grava um resumo em log_listagem_volume."""
    resumo = []
    for pasta in PASTAS:
        try:
            arquivos = sorted((f for f in dbutils.fs.ls(f"{BASE}/{pasta}") if f.name.endswith(".json")),
                              key=lambda f: f.name)
        except Exception:
            arquivos = []
        print(f"\n=== {BASE}/{pasta}  ({len(arquivos)} arquivos) ===")
        for f in arquivos:
            print(f"{f.name:<45} {f.size:>10} bytes")
        resumo.append((momento, pasta, len(arquivos),
                       arquivos[0].name if arquivos else None,
                       arquivos[-1].name if arquivos else None))
    df = (spark.createDataFrame(resumo, "momento string, pasta string, qtd_arquivos int, "
                                        "primeiro_arquivo string, ultimo_arquivo string")
          .withColumn("registrado_em", F.current_timestamp()))
    df.write.mode("append").saveAsTable("workspace.gs.log_listagem_volume")
    display(df.drop("registrado_em"))


listar_volume(dbutils.widgets.get("momento"))


=== /Volumes/workspace/gs/chegada/clima  (75 arquivos) ===
clima_3509502_2026-10-25.json                       1229 bytes
clima_3509502_2026-10-26.json                       1229 bytes
clima_3509502_2026-10-27.json                       1229 bytes
clima_3509502_2026-10-28.json                       1229 bytes
clima_3509502_2026-10-29.json                       1230 bytes
clima_3509502_2026-10-30.json                       1230 bytes
clima_3509502_2026-10-31.json                       1230 bytes
clima_3509502_2026-11-01.json                       1229 bytes
clima_3509502_2026-11-02.json                       1229 bytes
clima_3509502_2026-11-03.json                       1230 bytes
clima_3509502_2026-11-04.json                       1229 bytes
clima_3509502_2026-11-05.json                       1230 bytes
clima_3509502_2026-11-06.json                       1229 bytes
clima_3509502_2026-11-07.json                       1229 bytes
clima_3509502_2026-11-08.json                       1229 b

momento,pasta,qtd_arquivos,primeiro_arquivo,ultimo_arquivo
3_reexecucao_sem_arquivo_novo,clima,75,clima_3509502_2026-10-25.json,clima_3543402_2026-11-18.json
3_reexecucao_sem_arquivo_novo,ibge_municipios,25,ibge_municipios_2026-10-25.json,ibge_municipios_2026-11-18.json
3_reexecucao_sem_arquivo_novo,lavouras,25,lavouras_2026-10-25.json,lavouras_2026-11-18.json


### Histórico de listagens (requisito 4: Volume depois de cada lote)

In [0]:
display(spark.table("workspace.gs.log_listagem_volume").orderBy("registrado_em", "pasta"))

momento,pasta,qtd_arquivos,primeiro_arquivo,ultimo_arquivo,registrado_em
1_apos_lote1,clima,39,clima_3509502_2026-10-25.json,clima_3543402_2026-11-06.json,2026-11-18T13:11:15.981Z
1_apos_lote1,ibge_municipios,13,ibge_municipios_2026-10-25.json,ibge_municipios_2026-11-06.json,2026-11-18T13:11:15.981Z
1_apos_lote1,lavouras,13,lavouras_2026-10-25.json,lavouras_2026-11-06.json,2026-11-18T13:11:15.981Z
2_apos_lote2,clima,75,clima_3509502_2026-10-25.json,clima_3543402_2026-11-18.json,2026-11-18T14:28:52.309Z
2_apos_lote2,ibge_municipios,25,ibge_municipios_2026-10-25.json,ibge_municipios_2026-11-18.json,2026-11-18T14:28:52.309Z
2_apos_lote2,lavouras,25,lavouras_2026-10-25.json,lavouras_2026-11-18.json,2026-11-18T14:28:52.309Z


# AgroFácil · 01 · Camada Bronze (Auto Loader, incremental e sem duplicação)

* Uma tabela Bronze por fonte: `bronze_clima`, `bronze_ibge_municipios`, `bronze_lavouras`.
* Cópia fiel: nada é filtrado nem corrigido. Entram só `arquivo_origem` (`_metadata.file_path`) e `carregado_em`.
A única mudança no conteúdo é trocar `-` por `_` em nomes de coluna (o IBGE traz `regiao-imediata`).
* Sem `inferColumnTypes`, os valores chegam como texto (objetos aninhados viram texto JSON). A tipagem é feita na Prata com `CAST`.
* `multiLine = true`: o JSON do IBGE e o das lavouras são listas de objetos.
* `trigger(availableNow=True)` + `awaitTermination()`: processa só os arquivos que o checkpoint ainda não viu e **termina antes da próxima célula**.
* Rode **depois de cada lote** mudando o widget `momento`. As contagens ficam gravadas em `log_contagens_bronze`.

In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/gs/chegada"
FONTES = { 
    "bronze_clima": "clima",
    "bronze_ibge_municipios": "ibge_municipios",
    "bronze_lavouras": "lavouras",
}
dbutils.widgets.text("momento", "1_apos_lote1", "Momento da contagem")
MOMENTO = dbutils.widgets.get("momento")


def carregar_bronze(tabela, pasta):
    df = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", f"{BASE}/_controle/schema_{tabela}")
          .option("multiLine", "true")
          .load(f"{BASE}/{pasta}")
          .withColumn("arquivo_origem", F.col("_metadata.file_path"))
          .withColumn("carregado_em", F.current_timestamp()))
    for c in df.columns:                      
        if "-" in c:
            df = df.withColumnRenamed(c, c.replace("-", "_"))
    (df.writeStream
       .option("checkpointLocation", f"{BASE}/_controle/checkpoint_{tabela}")
       .trigger(availableNow=True)
       .toTable(f"workspace.gs.{tabela}")
       ).awaitTermination()
    print(f"OK {tabela} <- {pasta}/")


for tabela, pasta in FONTES.items():
    carregar_bronze(tabela, pasta)

OK bronze_clima <- clima/
OK bronze_ibge_municipios <- ibge_municipios/
OK bronze_lavouras <- lavouras/


### Contagens por arquivo e total (evidência dos 3 momentos)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gs.log_contagens_bronze (
  momento STRING, tabela STRING, total_linhas BIGINT, arquivos_distintos BIGINT,
  linhas_adicionadas BIGINT, registrado_em TIMESTAMP
) USING DELTA
""")

log = []
for tabela in FONTES:
    df = spark.table(f"workspace.gs.{tabela}")
    print(f"\n##### {tabela} · momento: {MOMENTO}")
    display(df.groupBy(F.element_at(F.split("arquivo_origem", "/"), -1).alias("arquivo"))
              .count().withColumnRenamed("count", "linhas").orderBy("arquivo"))
    total = df.count()
    arquivos = df.select("arquivo_origem").distinct().count()
    anterior = (spark.table("workspace.gs.log_contagens_bronze").filter(F.col("tabela") == tabela)
                .orderBy(F.col("registrado_em").desc()).limit(1).collect())
    adicionadas = total - (anterior[0]["total_linhas"] if anterior else 0)
    log.append((MOMENTO, tabela, total, arquivos, adicionadas))

(spark.createDataFrame(log, "momento string, tabela string, total_linhas long, arquivos_distintos long, "
                            "linhas_adicionadas long")
    .withColumn("registrado_em", F.current_timestamp())
    .write.mode("append").saveAsTable("workspace.gs.log_contagens_bronze"))


##### bronze_clima · momento: 3_reexecucao_sem_arquivo_novo


arquivo,linhas
clima_3509502_2026-10-25.json,1
clima_3509502_2026-10-26.json,1
clima_3509502_2026-10-27.json,1
clima_3509502_2026-10-28.json,1
clima_3509502_2026-10-29.json,1
clima_3509502_2026-10-30.json,1
clima_3509502_2026-10-31.json,1
clima_3509502_2026-11-01.json,1
clima_3509502_2026-11-02.json,1
clima_3509502_2026-11-03.json,1



##### bronze_ibge_municipios · momento: 3_reexecucao_sem_arquivo_novo


arquivo,linhas
ibge_municipios_2026-10-25.json,645
ibge_municipios_2026-10-26.json,645
ibge_municipios_2026-10-27.json,645
ibge_municipios_2026-10-28.json,645
ibge_municipios_2026-10-29.json,645
ibge_municipios_2026-10-30.json,645
ibge_municipios_2026-10-31.json,645
ibge_municipios_2026-11-01.json,645
ibge_municipios_2026-11-02.json,645
ibge_municipios_2026-11-03.json,645



##### bronze_lavouras · momento: 3_reexecucao_sem_arquivo_novo


arquivo,linhas
lavouras_2026-10-25.json,10
lavouras_2026-10-26.json,9
lavouras_2026-10-27.json,9
lavouras_2026-10-28.json,9
lavouras_2026-10-29.json,9
lavouras_2026-10-30.json,13
lavouras_2026-10-31.json,9
lavouras_2026-11-01.json,9
lavouras_2026-11-02.json,9
lavouras_2026-11-03.json,9


### Resumo dos momentos
Depois do 1º lote entram só os arquivos do lote 1; depois do 2º, só os do lote 2; na reexecução sem arquivo novo `linhas_adicionadas` deve ser **0**.

In [0]:
display(spark.table("workspace.gs.log_contagens_bronze").orderBy("registrado_em", "tabela"))

momento,tabela,total_linhas,arquivos_distintos,linhas_adicionadas,registrado_em
1_apos_lote1,bronze_clima,39,39,39,2026-11-18T13:15:14.303Z
1_apos_lote1,bronze_ibge_municipios,8385,13,8385,2026-11-18T13:15:14.303Z
1_apos_lote1,bronze_lavouras,126,13,126,2026-11-18T13:15:14.303Z
2_apos_lote2,bronze_clima,75,75,36,2026-11-18T14:30:37.695Z
2_apos_lote2,bronze_ibge_municipios,16125,25,7740,2026-11-18T14:30:37.695Z
2_apos_lote2,bronze_lavouras,242,25,116,2026-11-18T14:30:37.695Z
3_reexecucao_sem_arquivo_novo,bronze_clima,75,75,0,2026-11-18T14:33:04.371Z
3_reexecucao_sem_arquivo_novo,bronze_ibge_municipios,16125,25,0,2026-11-18T14:33:04.371Z
3_reexecucao_sem_arquivo_novo,bronze_lavouras,242,25,0,2026-11-18T14:33:04.371Z


### Histórico Delta (evidência alternativa: `DESCRIBE HISTORY`)

In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.gs.bronze_clima")
        .select("version", "timestamp", "operation", "operationMetrics"))

version,timestamp,operation,operationMetrics
3,2026-11-18T14:29:48.909Z,STREAMING UPDATE,"Map(numAddedFiles -> 1, numOutputBytes -> 14227, numOutputRows -> 36, numRemovedFiles -> 0)"
2,2026-11-18T13:14:05.151Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 76, numOutputRows -> 29, numOutputBytes -> 13093, numAddedFiles -> 1)"
1,2026-11-18T13:14:03.992Z,STREAMING UPDATE,"Map(numAddedFiles -> 1, numOutputBytes -> 10189, numOutputRows -> 10, numRemovedFiles -> 0)"
0,2026-11-18T13:13:54.930Z,CREATE TABLE,Map()


### Conferência dos formatos (rode uma vez após o lote 1)
A Prata depende destes nomes de coluna. Se algum faltar, este é o lugar de comparar com o arquivo real.

In [0]:
ESPERADO = {
    "bronze_clima": ["hourly", "latitude", "longitude", "arquivo_origem", "carregado_em"],
    "bronze_ibge_municipios": ["id", "nome", "microrregiao", "arquivo_origem", "carregado_em"],
    "bronze_lavouras": ["codigo_ibge", "municipio", "cultura", "area_plantada_ha", "producao_toneladas",
                        "ano_referencia", "arquivo_origem", "carregado_em"],
}
for t, cols in ESPERADO.items():
    atuais = spark.table(f"workspace.gs.{t}").columns
    faltando = [c for c in cols if c not in atuais]
    print(f"{t}: {'OK' if not faltando else 'FALTAM ' + str(faltando)} | colunas: {atuais}")
display(spark.table("workspace.gs.bronze_lavouras").limit(5))

bronze_clima: OK | colunas: ['elevation', 'generationtime_ms', 'hourly', 'hourly_units', 'latitude', 'longitude', 'timezone', 'timezone_abbreviation', 'utc_offset_seconds', '_rescued_data', 'arquivo_origem', 'carregado_em']
bronze_ibge_municipios: OK | colunas: ['id', 'microrregiao', 'nome', 'regiao_imediata', '_rescued_data', 'arquivo_origem', 'carregado_em']
bronze_lavouras: OK | colunas: ['ano_referencia', 'area_plantada_ha', 'codigo_ibge', 'cultura', 'municipio', 'origem', 'producao_toneladas', 'uf', '_rescued_data', 'arquivo_origem', 'carregado_em']


ano_referencia,area_plantada_ha,codigo_ibge,cultura,municipio,origem,producao_toneladas,uf,_rescued_data,arquivo_origem,carregado_em
2025,9531.4,3509502,Cana-de-açúcar,Campinas,simulado,744161.8,SP,null,/Volumes/workspace/gs/chegada/lavouras/lavouras_2026-11-12.json,2026-11-18T14:30:04.842Z
2025,2986.8,3509502,Milho,Campinas,simulado,18519.2,SP,null,/Volumes/workspace/gs/chegada/lavouras/lavouras_2026-11-12.json,2026-11-18T14:30:04.842Z
2025,1872.7,3509502,Soja,Campinas,simulado,6439.7,SP,null,/Volumes/workspace/gs/chegada/lavouras/lavouras_2026-11-12.json,2026-11-18T14:30:04.842Z
2025,52478.0,3543402,Cana-de-açúcar,Ribeirão Preto,simulado,4184339.0,SP,null,/Volumes/workspace/gs/chegada/lavouras/lavouras_2026-11-12.json,2026-11-18T14:30:04.842Z
2025,6522.9,3543402,Soja,Ribeirão Preto,simulado,23022.2,SP,null,/Volumes/workspace/gs/chegada/lavouras/lavouras_2026-11-12.json,2026-11-18T14:30:04.842Z


# AgroFácil · 02 · Camada Prata

Tipagem estrita, nulos e inválidos tratados, **uma linha por chave**, `MERGE` idempotente e regras de qualidade com a contagem de linhas barradas.

| Tabela Prata | Chave (o que identifica uma linha) | Origem |
| :-- | :-- | :-- |
| `prata_ibge_municipios` | `codigo_ibge` | cadastro real do IBGE (municípios de SP) |
| `prata_clima` | `codigo_ibge` + `data_hora` (um município por hora) | Open-Meteo |
| `prata_lavouras` | `codigo_ibge` + `ano_referencia` + `cultura` | cadastro de lavouras **simulado** |

Ordem: municípios (cadastro) -> clima -> lavouras -> tabela de regras de qualidade.
O município do clima vem do **nome do arquivo** (`clima_<codigo_ibge>_<ds>.json`), nunca de um valor fixo.
Em todo `QUALIFY`, empate de `carregado_em` (arquivos do mesmo lote) é desfeito por `arquivo_origem DESC`: vale o arquivo de data mais recente.

In [0]:
from pyspark.sql import functions as F

def normalizar(tabela):
    df = spark.table(f"workspace.gs.{tabela}")
    for c, t in df.dtypes:
        if t.startswith("struct") or t.startswith("array"):
            df = df.withColumn(c, F.to_json(c))
    df.createOrReplaceTempView("v_" + tabela)
    return df

esperado = {
    "bronze_clima": ["hourly", "arquivo_origem", "carregado_em"],
    "bronze_ibge_municipios": ["id", "nome", "microrregiao", "arquivo_origem", "carregado_em"],
    "bronze_lavouras": ["codigo_ibge", "cultura", "area_plantada_ha", "producao_toneladas",
                        "ano_referencia", "arquivo_origem", "carregado_em"],
}
for tabela, cols in esperado.items():
    df = normalizar(tabela)
    faltando = [c for c in cols if c not in df.columns]
    if faltando:
        raise Exception(f"{tabela}: faltam as colunas {faltando}. Colunas reais: {df.columns}.")
    print(f"OK {tabela}: {df.count()} linhas")

OK bronze_clima: 75 linhas
OK bronze_ibge_municipios: 16125 linhas
OK bronze_lavouras: 242 linhas


## 1. `prata_ibge_municipios` (cadastro oficial de municípios)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.gs.prata_ibge_municipios (
  codigo_ibge   STRING,
  municipio     STRING,
  uf            STRING,
  microrregiao  STRING,
  mesorregiao   STRING,
  atualizado_em TIMESTAMP
) USING DELTA;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_municipios AS
SELECT
  TRIM(CAST(id AS STRING))                                        AS codigo_ibge,
  INITCAP(TRIM(nome))                                             AS municipio,
  UPPER(TRIM(get_json_object(microrregiao, '$.mesorregiao.UF.sigla'))) AS uf,
  TRIM(get_json_object(microrregiao, '$.nome'))                   AS microrregiao,
  TRIM(get_json_object(microrregiao, '$.mesorregiao.nome'))       AS mesorregiao,
  carregado_em,
  arquivo_origem
FROM v_bronze_ibge_municipios
WHERE id IS NOT NULL
  AND LENGTH(TRIM(CAST(id AS STRING))) = 7
  AND nome IS NOT NULL AND TRIM(nome) <> ''
QUALIFY ROW_NUMBER() OVER (PARTITION BY TRIM(CAST(id AS STRING)) ORDER BY carregado_em DESC, arquivo_origem DESC) = 1;

CREATE OR REPLACE TEMP VIEW qualidade_municipios AS
SELECT 'M1' AS regra, 'prata_ibge_municipios' AS tabela, 'código IBGE nulo ou diferente de 7 dígitos' AS motivo,
       COUNT_IF(id IS NULL OR LENGTH(TRIM(CAST(id AS STRING))) <> 7) AS linhas_barradas
FROM v_bronze_ibge_municipios
UNION ALL
SELECT 'M2', 'prata_ibge_municipios', 'nome do município nulo ou vazio',
       COUNT_IF(nome IS NULL OR TRIM(nome) = '')
FROM v_bronze_ibge_municipios;

In [0]:
%sql

-- MERGE 1ª execução
MERGE INTO workspace.gs.prata_ibge_municipios AS alvo
USING stg_municipios AS novo
ON alvo.codigo_ibge = novo.codigo_ibge
WHEN MATCHED AND NOT (alvo.municipio <=> novo.municipio AND alvo.uf <=> novo.uf
                      AND alvo.microrregiao <=> novo.microrregiao AND alvo.mesorregiao <=> novo.mesorregiao)
  THEN UPDATE SET alvo.municipio = novo.municipio, alvo.uf = novo.uf, alvo.microrregiao = novo.microrregiao,
                  alvo.mesorregiao = novo.mesorregiao, alvo.atualizado_em = current_timestamp()
WHEN NOT MATCHED
  THEN INSERT (codigo_ibge, municipio, uf, microrregiao, mesorregiao, atualizado_em)
       VALUES (novo.codigo_ibge, novo.municipio, novo.uf, novo.microrregiao, novo.mesorregiao, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
645,0,0,645


In [0]:
%sql

-- MERGE 2ª execução (idempotência: num_inserted_rows = 0 e num_updated_rows = 0)
MERGE INTO workspace.gs.prata_ibge_municipios AS alvo
USING stg_municipios AS novo
ON alvo.codigo_ibge = novo.codigo_ibge
WHEN MATCHED AND NOT (alvo.municipio <=> novo.municipio AND alvo.uf <=> novo.uf
                      AND alvo.microrregiao <=> novo.microrregiao AND alvo.mesorregiao <=> novo.mesorregiao)
  THEN UPDATE SET alvo.municipio = novo.municipio, alvo.uf = novo.uf, alvo.microrregiao = novo.microrregiao,
                  alvo.mesorregiao = novo.mesorregiao, alvo.atualizado_em = current_timestamp()
WHEN NOT MATCHED
  THEN INSERT (codigo_ibge, municipio, uf, microrregiao, mesorregiao, atualizado_em)
       VALUES (novo.codigo_ibge, novo.municipio, novo.uf, novo.microrregiao, novo.mesorregiao, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

SELECT COUNT(*) AS municipios_na_prata, COUNT(DISTINCT codigo_ibge) AS chaves_distintas,
       COUNT_IF(codigo_ibge IN ('3509502', '3543402', '3538709')) AS monitorados_encontrados
FROM workspace.gs.prata_ibge_municipios;

municipios_na_prata,chaves_distintas,monitorados_encontrados
645,645,3


## 2. `prata_clima` (listas paralelas -> uma linha por município por hora)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.gs.prata_clima (
  codigo_ibge   STRING,
  data_hora     TIMESTAMP,
  temperatura_c DOUBLE,
  chuva_mm      DOUBLE,
  umidade_solo  DOUBLE,
  atualizado_em TIMESTAMP
) USING DELTA;

In [0]:
%sql

-- arrays_zip junta as listas paralelas da Open-Meteo; explode abre em uma linha por horário.
-- O código do município vem do nome do arquivo: clima_<codigo_ibge>_<ds>.json (arquivo fora do padrão fica sem código e é barrado por C6).
CREATE OR REPLACE TEMP VIEW clima_horario AS
SELECT
  REGEXP_EXTRACT(h.arquivo_origem, 'clima_([0-9]{7})_', 1)  AS codigo_ibge,
  TO_TIMESTAMP(h.z.time)                                     AS data_hora,
  CAST(h.z.temperature_2m AS DOUBLE)                         AS temperatura_c,
  CAST(h.z.precipitation AS DOUBLE)                          AS chuva_mm,
  CAST(h.z.soil_moisture_0_to_7cm AS DOUBLE)                 AS umidade_solo,
  (m.codigo_ibge IS NOT NULL)                                AS no_cadastro,
  h.carregado_em,
  h.arquivo_origem
FROM (
  SELECT EXPLODE(ARRAYS_ZIP(p.time, p.temperature_2m, p.precipitation, p.soil_moisture_0_to_7cm)) AS z,
         arquivo_origem, carregado_em
  FROM (
    SELECT FROM_JSON(hourly, 'time ARRAY<STRING>, temperature_2m ARRAY<DOUBLE>, precipitation ARRAY<DOUBLE>, soil_moisture_0_to_7cm ARRAY<DOUBLE>') AS p,
           arquivo_origem, carregado_em
    FROM v_bronze_clima
  )
) h
LEFT JOIN workspace.gs.prata_ibge_municipios m
  ON REGEXP_EXTRACT(h.arquivo_origem, 'clima_([0-9]{7})_', 1) = m.codigo_ibge;

-- Regras de qualidade: quantas linhas cada regra barra (uma linha pode violar mais de uma regra)
CREATE OR REPLACE TEMP VIEW qualidade_clima AS
SELECT 'C1' AS regra, 'prata_clima' AS tabela, 'data/hora nula' AS motivo,
       COUNT_IF(data_hora IS NULL) AS linhas_barradas FROM clima_horario
UNION ALL SELECT 'C2', 'prata_clima', 'chuva negativa (chuva_mm < 0)',
       COUNT_IF(chuva_mm < 0) FROM clima_horario
UNION ALL SELECT 'C3', 'prata_clima', 'temperatura fora de -10 a 50 °C',
       COUNT_IF(temperatura_c < -10 OR temperatura_c > 50) FROM clima_horario
UNION ALL SELECT 'C4', 'prata_clima', 'umidade do solo fora de 0 a 1 m³/m³',
       COUNT_IF(umidade_solo < 0 OR umidade_solo > 1) FROM clima_horario
UNION ALL SELECT 'C5', 'prata_clima', 'leitura nula (temperatura, chuva ou umidade)',
       COUNT_IF(temperatura_c IS NULL OR chuva_mm IS NULL OR umidade_solo IS NULL) FROM clima_horario
UNION ALL SELECT 'C6', 'prata_clima', 'município fora do cadastro do IBGE (ou arquivo fora do padrão de nome)',
       COUNT_IF(NOT no_cadastro) FROM clima_horario;

-- Lote limpo: uma linha por chave (município + data/hora); em duplicata vale a carga mais recente
CREATE OR REPLACE TEMP VIEW lote_prata_clima AS
SELECT codigo_ibge, data_hora, temperatura_c, chuva_mm, umidade_solo
FROM clima_horario
WHERE data_hora IS NOT NULL
  AND chuva_mm >= 0
  AND temperatura_c BETWEEN -10 AND 50
  AND umidade_solo BETWEEN 0 AND 1
  AND no_cadastro
QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_ibge, data_hora ORDER BY carregado_em DESC, arquivo_origem DESC) = 1;

In [0]:
%sql

SELECT * FROM qualidade_clima ORDER BY regra;

regra,tabela,motivo,linhas_barradas
C1,prata_clima,data/hora nula,0
C2,prata_clima,chuva negativa (chuva_mm < 0),0
C3,prata_clima,temperatura fora de -10 a 50 °C,0
C4,prata_clima,umidade do solo fora de 0 a 1 m³/m³,0
C5,prata_clima,"leitura nula (temperatura, chuva ou umidade)",0
C6,prata_clima,município fora do cadastro do IBGE (ou arquivo fora do padrão de nome),0


In [0]:
%sql

-- MERGE 1ª execução
MERGE INTO workspace.gs.prata_clima AS alvo
USING lote_prata_clima AS novo
ON alvo.codigo_ibge = novo.codigo_ibge AND alvo.data_hora = novo.data_hora
WHEN MATCHED AND NOT (alvo.temperatura_c <=> novo.temperatura_c AND alvo.chuva_mm <=> novo.chuva_mm
                      AND alvo.umidade_solo <=> novo.umidade_solo)
  THEN UPDATE SET alvo.temperatura_c = novo.temperatura_c, alvo.chuva_mm = novo.chuva_mm,
                  alvo.umidade_solo = novo.umidade_solo, alvo.atualizado_em = current_timestamp()
WHEN NOT MATCHED
  THEN INSERT (codigo_ibge, data_hora, temperatura_c, chuva_mm, umidade_solo, atualizado_em)
       VALUES (novo.codigo_ibge, novo.data_hora, novo.temperatura_c, novo.chuva_mm, novo.umidade_solo, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1800,0,0,1800


In [0]:
%sql

-- MERGE 2ª execução (idempotência: num_inserted_rows = 0 e num_updated_rows = 0)
MERGE INTO workspace.gs.prata_clima AS alvo
USING lote_prata_clima AS novo
ON alvo.codigo_ibge = novo.codigo_ibge AND alvo.data_hora = novo.data_hora
WHEN MATCHED AND NOT (alvo.temperatura_c <=> novo.temperatura_c AND alvo.chuva_mm <=> novo.chuva_mm
                      AND alvo.umidade_solo <=> novo.umidade_solo)
  THEN UPDATE SET alvo.temperatura_c = novo.temperatura_c, alvo.chuva_mm = novo.chuva_mm,
                  alvo.umidade_solo = novo.umidade_solo, alvo.atualizado_em = current_timestamp()
WHEN NOT MATCHED
  THEN INSERT (codigo_ibge, data_hora, temperatura_c, chuva_mm, umidade_solo, atualizado_em)
       VALUES (novo.codigo_ibge, novo.data_hora, novo.temperatura_c, novo.chuva_mm, novo.umidade_solo, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

-- Total da Prata e prova de que a chave é única (esperado: 24 linhas por município por dia coletado)
SELECT COUNT(*) AS linhas_prata_clima,
       COUNT(DISTINCT codigo_ibge, data_hora) AS chaves_distintas,
       COUNT(DISTINCT codigo_ibge) AS municipios,
       COUNT(DISTINCT TO_DATE(data_hora)) AS dias,
       MIN(data_hora) AS primeira_hora, MAX(data_hora) AS ultima_hora
FROM workspace.gs.prata_clima;

linhas_prata_clima,chaves_distintas,municipios,dias,primeira_hora,ultima_hora
1800,1800,3,25,2026-10-25T00:00:00.000Z,2026-11-18T23:00:00.000Z


## 3. `prata_lavouras` (cadastro de lavouras simulado)

Fonte **simulada** (gerador em `dags/agrofacil_coleta_dados.py`, função `gerar_lavouras`). Alguns dias trazem linhas sujas de propósito
(área zero, produção negativa, município inexistente e duplicata) para exercitar as regras abaixo.

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.gs.prata_lavouras (
  codigo_ibge          STRING,
  ano_referencia       INT,
  cultura              STRING,
  area_plantada_ha     DOUBLE,
  producao_toneladas   DOUBLE,
  produtividade_ton_ha DOUBLE,
  atualizado_em        TIMESTAMP
) USING DELTA;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW lavouras_linhas AS
SELECT
  TRIM(b.codigo_ibge)                        AS codigo_ibge,
  TRY_CAST(TRIM(b.ano_referencia) AS INT)    AS ano_referencia,
  TRIM(b.cultura)                            AS cultura,
  TRY_CAST(TRIM(b.area_plantada_ha) AS DOUBLE)   AS area_plantada_ha,
  TRY_CAST(TRIM(b.producao_toneladas) AS DOUBLE) AS producao_toneladas,
  (m.codigo_ibge IS NOT NULL)                AS no_cadastro,
  b.carregado_em,
  b.arquivo_origem
FROM v_bronze_lavouras b
LEFT JOIN workspace.gs.prata_ibge_municipios m ON TRIM(b.codigo_ibge) = m.codigo_ibge;

CREATE OR REPLACE TEMP VIEW qualidade_lavouras AS
SELECT 'L1' AS regra, 'prata_lavouras' AS tabela, 'área plantada nula, zero ou negativa' AS motivo,
       COUNT_IF(area_plantada_ha IS NULL OR area_plantada_ha <= 0) AS linhas_barradas FROM lavouras_linhas
UNION ALL SELECT 'L2', 'prata_lavouras', 'produção nula ou negativa',
       COUNT_IF(producao_toneladas IS NULL OR producao_toneladas < 0) FROM lavouras_linhas
UNION ALL SELECT 'L3', 'prata_lavouras', 'município fora do cadastro do IBGE',
       COUNT_IF(NOT no_cadastro) FROM lavouras_linhas
UNION ALL SELECT 'L4', 'prata_lavouras', 'produtividade implausível (fora de 0,5 a 120 t/ha)',
       COUNT_IF(area_plantada_ha > 0 AND producao_toneladas >= 0
                AND (producao_toneladas / area_plantada_ha < 0.5 OR producao_toneladas / area_plantada_ha > 120)) FROM lavouras_linhas;

-- Lote limpo: uma linha por chave (município + ano + cultura)
CREATE OR REPLACE TEMP VIEW lote_prata_lavouras AS
SELECT codigo_ibge, ano_referencia, cultura, area_plantada_ha, producao_toneladas,
       ROUND(producao_toneladas / area_plantada_ha, 2) AS produtividade_ton_ha
FROM lavouras_linhas
WHERE ano_referencia IS NOT NULL AND cultura IS NOT NULL AND cultura <> ''
  AND area_plantada_ha > 0
  AND producao_toneladas >= 0
  AND no_cadastro
  AND producao_toneladas / area_plantada_ha BETWEEN 0.5 AND 120
QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_ibge, ano_referencia, cultura
                           ORDER BY carregado_em DESC, arquivo_origem DESC) = 1;

In [0]:
%sql

SELECT * FROM qualidade_lavouras ORDER BY regra;

regra,tabela,motivo,linhas_barradas
L1,prata_lavouras,"área plantada nula, zero ou negativa",4
L2,prata_lavouras,produção nula ou negativa,4
L3,prata_lavouras,município fora do cadastro do IBGE,4
L4,prata_lavouras,"produtividade implausível (fora de 0,5 a 120 t/ha)",0


In [0]:
%sql

-- MERGE 1ª execução
MERGE INTO workspace.gs.prata_lavouras AS alvo
USING lote_prata_lavouras AS novo
ON alvo.codigo_ibge = novo.codigo_ibge AND alvo.ano_referencia = novo.ano_referencia AND alvo.cultura = novo.cultura
WHEN MATCHED AND NOT (alvo.area_plantada_ha <=> novo.area_plantada_ha
                      AND alvo.producao_toneladas <=> novo.producao_toneladas)
  THEN UPDATE SET alvo.area_plantada_ha = novo.area_plantada_ha, alvo.producao_toneladas = novo.producao_toneladas,
                  alvo.produtividade_ton_ha = novo.produtividade_ton_ha, alvo.atualizado_em = current_timestamp()
WHEN NOT MATCHED
  THEN INSERT (codigo_ibge, ano_referencia, cultura, area_plantada_ha, producao_toneladas, produtividade_ton_ha, atualizado_em)
       VALUES (novo.codigo_ibge, novo.ano_referencia, novo.cultura, novo.area_plantada_ha, novo.producao_toneladas,
               novo.produtividade_ton_ha, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
9,0,0,9


In [0]:
%sql

-- MERGE 2ª execução (idempotência: nada inserido, nada atualizado)
MERGE INTO workspace.gs.prata_lavouras AS alvo
USING lote_prata_lavouras AS novo
ON alvo.codigo_ibge = novo.codigo_ibge AND alvo.ano_referencia = novo.ano_referencia AND alvo.cultura = novo.cultura
WHEN MATCHED AND NOT (alvo.area_plantada_ha <=> novo.area_plantada_ha
                      AND alvo.producao_toneladas <=> novo.producao_toneladas)
  THEN UPDATE SET alvo.area_plantada_ha = novo.area_plantada_ha, alvo.producao_toneladas = novo.producao_toneladas,
                  alvo.produtividade_ton_ha = novo.produtividade_ton_ha, alvo.atualizado_em = current_timestamp()
WHEN NOT MATCHED
  THEN INSERT (codigo_ibge, ano_referencia, cultura, area_plantada_ha, producao_toneladas, produtividade_ton_ha, atualizado_em)
       VALUES (novo.codigo_ibge, novo.ano_referencia, novo.cultura, novo.area_plantada_ha, novo.producao_toneladas,
               novo.produtividade_ton_ha, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

SELECT l.*, m.municipio
FROM workspace.gs.prata_lavouras l
JOIN workspace.gs.prata_ibge_municipios m ON m.codigo_ibge = l.codigo_ibge
ORDER BY m.municipio, l.cultura;

codigo_ibge,ano_referencia,cultura,area_plantada_ha,producao_toneladas,produtividade_ton_ha,atualizado_em,municipio
3509502,2025,Cana-de-açúcar,9531.4,744161.8,78.07,2026-11-18T14:39:30.936Z,Campinas
3509502,2025,Milho,2986.8,18519.2,6.2,2026-11-18T14:39:30.936Z,Campinas
3509502,2025,Soja,1872.7,6439.7,3.44,2026-11-18T14:39:30.936Z,Campinas
3538709,2025,Cana-de-açúcar,41472.9,3124490.1,75.34,2026-11-18T14:39:30.936Z,Piracicaba
3538709,2025,Laranja,1232.8,35804.3,29.04,2026-11-18T14:39:30.936Z,Piracicaba
3538709,2025,Milho,5050.3,32973.5,6.53,2026-11-18T14:39:30.936Z,Piracicaba
3543402,2025,Cana-de-açúcar,52478,4184339,79.74,2026-11-18T14:39:30.936Z,Ribeirão Preto
3543402,2025,Milho,4247.6,30326.3,7.14,2026-11-18T14:39:30.936Z,Ribeirão Preto
3543402,2025,Soja,6522.9,23022.2,3.53,2026-11-18T14:39:30.936Z,Ribeirão Preto


## 4. Tabela de regras de qualidade (regra, motivo, linhas barradas)
Contagem sobre as linhas da Bronze (se o mesmo registro sujo chega em vários arquivos, ele conta uma vez por arquivo).

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.gs.prata_qualidade_regras AS
SELECT current_timestamp() AS executado_em, regra, tabela, motivo, linhas_barradas
FROM (
  SELECT * FROM qualidade_municipios
  UNION ALL SELECT * FROM qualidade_clima
  UNION ALL SELECT * FROM qualidade_lavouras
);

SELECT regra, tabela, motivo, linhas_barradas FROM workspace.gs.prata_qualidade_regras ORDER BY regra;

regra,tabela,motivo,linhas_barradas
C1,prata_clima,data/hora nula,0
C2,prata_clima,chuva negativa (chuva_mm < 0),0
C3,prata_clima,temperatura fora de -10 a 50 °C,0
C4,prata_clima,umidade do solo fora de 0 a 1 m³/m³,0
C5,prata_clima,"leitura nula (temperatura, chuva ou umidade)",0
C6,prata_clima,município fora do cadastro do IBGE (ou arquivo fora do padrão de nome),0
L1,prata_lavouras,"área plantada nula, zero ou negativa",4
L2,prata_lavouras,produção nula ou negativa,4
L3,prata_lavouras,município fora do cadastro do IBGE,4
L4,prata_lavouras,"produtividade implausível (fora de 0,5 a 120 t/ha)",0


In [0]:
%sql

-- Total de linhas de cada tabela Prata
SELECT 'prata_ibge_municipios' AS tabela, COUNT(*) AS total_linhas FROM workspace.gs.prata_ibge_municipios
UNION ALL SELECT 'prata_clima', COUNT(*) FROM workspace.gs.prata_clima
UNION ALL SELECT 'prata_lavouras', COUNT(*) FROM workspace.gs.prata_lavouras;

tabela,total_linhas
prata_ibge_municipios,645
prata_clima,1800
prata_lavouras,9


# AgroFácil · 03 · Camada Ouro (star schema) e contrato com o MVP

**Grão do fato:** uma linha por município por hora (`ouro_fato_clima_lavoura`).

| Tabela | Papel | Chave |
| :-- | :-- | :-- |
| `ouro_fato_clima_lavoura` | Fato: temperatura, chuva, umidade do solo e alerta | (`id_local`, `id_tempo`) |
| `ouro_dim_tempo` | Dimensão tempo: data, hora, dia da semana, mês, ano | `id_tempo` (AAAAMMDDHH) |
| `ouro_dim_local` | Dimensão local: município, UF, região, cultura principal, área plantada (lavouras simuladas) | `id_local` (código IBGE) |

O fato guarda as chaves e as métricas; o contexto (nome, região, lavoura) mora nas dimensões.
A fonte climática e a agrícola se cruzam pelo **código IBGE do município**.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.gs.ouro_dim_tempo AS
SELECT DISTINCT
  CAST(DATE_FORMAT(data_hora, 'yyyyMMddHH') AS INT)                                   AS id_tempo,
  TO_DATE(data_hora)                                                                  AS data,
  HOUR(data_hora)                                                                     AS hora,
  DAYOFWEEK(data_hora)                                                                AS dia_semana_num,
  ELT(DAYOFWEEK(data_hora), 'Domingo', 'Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado') AS dia_semana,
  MONTH(data_hora)                                                                    AS mes,
  YEAR(data_hora)                                                                     AS ano
FROM workspace.gs.prata_clima;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Dimensão local: só os municípios monitorados (os que têm clima), com o cadastro do IBGE
-- e o perfil agrícola do último ano do cadastro de lavouras (soma das culturas).
CREATE OR REPLACE TABLE workspace.gs.ouro_dim_local AS
WITH monitorados AS (
  SELECT DISTINCT codigo_ibge FROM workspace.gs.prata_clima
),
lav AS (
  SELECT codigo_ibge,
         MAX(ano_referencia)                          AS ano_lavoura,
         SUM(area_plantada_ha)                        AS area_plantada_total_ha,
         SUM(producao_toneladas)                      AS producao_total_ton,
         COUNT(DISTINCT cultura)                      AS qtd_culturas,
         MAX_BY(cultura, area_plantada_ha)            AS cultura_principal
  FROM workspace.gs.prata_lavouras
  WHERE ano_referencia = (SELECT MAX(ano_referencia) FROM workspace.gs.prata_lavouras)
  GROUP BY codigo_ibge
)
SELECT m.codigo_ibge AS id_local, m.municipio, m.uf, m.microrregiao, m.mesorregiao,
       l.ano_lavoura, l.area_plantada_total_ha, l.producao_total_ton, l.qtd_culturas, l.cultura_principal
FROM workspace.gs.prata_ibge_municipios m
JOIN monitorados mo ON mo.codigo_ibge = m.codigo_ibge
LEFT JOIN lav l ON l.codigo_ibge = m.codigo_ibge;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Fato: 1 linha por município por hora. Limiares do alerta: solo < 0,20 m³/m³ sem chuva; chuva >= 10 mm/h.
CREATE OR REPLACE TABLE workspace.gs.ouro_fato_clima_lavoura AS
SELECT
  l.id_local,
  t.id_tempo,
  p.temperatura_c,
  p.chuva_mm,
  p.umidade_solo,
  CASE
    WHEN p.umidade_solo < 0.20 AND p.chuva_mm = 0.0 THEN 'Alerta: irrigação necessária'
    WHEN p.chuva_mm >= 10.0                          THEN 'Alerta: chuva forte'
    ELSE 'Condição estável'
  END AS status_alerta
FROM workspace.gs.prata_clima p
JOIN workspace.gs.ouro_dim_local l ON l.id_local = p.codigo_ibge
JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = CAST(DATE_FORMAT(p.data_hora, 'yyyyMMddHH') AS INT);

num_affected_rows,num_inserted_rows


## Conferência de integridade
`ok = true` em todas as linhas: chaves únicas, nenhuma linha órfã e soma do fato igual à da Prata.

In [0]:
%sql

SELECT 'ouro_dim_tempo: id_tempo único' AS verificacao,
       CAST(COUNT(*) AS DOUBLE) AS valor_a, CAST(COUNT(DISTINCT id_tempo) AS DOUBLE) AS valor_b,
       COUNT(*) = COUNT(DISTINCT id_tempo) AS ok
FROM workspace.gs.ouro_dim_tempo
UNION ALL
SELECT 'ouro_dim_local: id_local único', CAST(COUNT(*) AS DOUBLE), CAST(COUNT(DISTINCT id_local) AS DOUBLE),
       COUNT(*) = COUNT(DISTINCT id_local)
FROM workspace.gs.ouro_dim_local
UNION ALL
SELECT 'fato: chave (id_local, id_tempo) única', CAST(COUNT(*) AS DOUBLE),
       CAST(COUNT(DISTINCT id_local, id_tempo) AS DOUBLE), COUNT(*) = COUNT(DISTINCT id_local, id_tempo)
FROM workspace.gs.ouro_fato_clima_lavoura
UNION ALL
SELECT 'fato: sem chaves órfãs (dim_local / dim_tempo)',
       CAST(COUNT(*) AS DOUBLE), 0.0, COUNT(*) = 0
FROM workspace.gs.ouro_fato_clima_lavoura f
LEFT JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
LEFT JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
WHERE l.id_local IS NULL OR t.id_tempo IS NULL
UNION ALL
SELECT 'soma da chuva: Prata = Fato (mm)',
       (SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.prata_clima),
       (SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.ouro_fato_clima_lavoura),
       (SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.prata_clima)
         = (SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.ouro_fato_clima_lavoura)
UNION ALL
SELECT 'linhas: Prata = Fato',
       (SELECT CAST(COUNT(*) AS DOUBLE) FROM workspace.gs.prata_clima),
       (SELECT CAST(COUNT(*) AS DOUBLE) FROM workspace.gs.ouro_fato_clima_lavoura),
       (SELECT COUNT(*) FROM workspace.gs.prata_clima) = (SELECT COUNT(*) FROM workspace.gs.ouro_fato_clima_lavoura);

verificacao,valor_a,valor_b,ok
ouro_dim_tempo: id_tempo único,600,600,true
ouro_dim_local: id_local único,3,3,true
"fato: chave (id_local, id_tempo) única",1800,1800,true
fato: sem chaves órfãs (dim_local / dim_tempo),0,0,true
soma da chuva: Prata = Fato (mm),432.6,432.6,true
linhas: Prata = Fato,1800,1800,true


## View de contrato com o MVP: `ouro_mvp_diario` (1 linha por município por dia)

Premissas do projeto:
* Dia seco: chuva do dia < 1 mm. Risco de estiagem: >= 10 dias secos seguidos = ALTO; >= 5 = MÉDIO.
* Necessidade hídrica de referência: **3,5 mm/dia**. **1 mm sobre 1 ha = 10 m³.**
* Água poupada no dia = área plantada (ha) x 10 x MIN(chuva do dia, 3,5 mm). Custo da água: **R$ 1,85/m³** (premissa, ajustável).
* Estresse térmico: horas com temperatura acima de 32 °C.

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.gs.ouro_mvp_diario AS
WITH dia AS (
  SELECT l.id_local, l.municipio, l.uf, l.mesorregiao, l.cultura_principal,
         COALESCE(l.area_plantada_total_ha, 0)                        AS area_plantada_ha,
         t.data,
         ROUND(SUM(f.chuva_mm), 1)                                    AS chuva_dia_mm,
         ROUND(AVG(f.temperatura_c), 1)                               AS temp_media_c,
         ROUND(MAX(f.temperatura_c), 1)                               AS temp_max_c,
         ROUND(AVG(f.umidade_solo), 3)                                AS umidade_solo_media,
         COUNT_IF(f.temperatura_c > 32)                               AS horas_estresse_termico
  FROM workspace.gs.ouro_fato_clima_lavoura f
  JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
  JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
  GROUP BY l.id_local, l.municipio, l.uf, l.mesorregiao, l.cultura_principal, l.area_plantada_total_ha, t.data
),
seq AS (
  SELECT *,
         CASE WHEN chuva_dia_mm < 1.0 THEN 1 ELSE 0 END AS dia_seco,
         DATEDIFF(data, DATE'2000-01-01')
           - ROW_NUMBER() OVER (PARTITION BY id_local, CASE WHEN chuva_dia_mm < 1.0 THEN 1 ELSE 0 END
                                ORDER BY data) AS grupo
  FROM dia
),
estiagem AS (
  SELECT *,
         CASE WHEN dia_seco = 1
              THEN ROW_NUMBER() OVER (PARTITION BY id_local, dia_seco, grupo ORDER BY data) ELSE 0 END AS dias_secos_consecutivos
  FROM seq
)
SELECT id_local, municipio, uf, mesorregiao, cultura_principal, area_plantada_ha, data,
       chuva_dia_mm, temp_media_c, temp_max_c, umidade_solo_media, horas_estresse_termico,
       CAST(dias_secos_consecutivos AS INT)                                           AS dias_secos_consecutivos,
       ROUND(chuva_dia_mm - 3.5, 1)                                                   AS saldo_hidrico_mm,
       CASE WHEN chuva_dia_mm < 2.0 AND (temp_media_c > 28.0 OR umidade_solo_media < 0.20) THEN 'ALTA'
            WHEN chuva_dia_mm < 5.0 THEN 'MODERADA'
            ELSE 'BAIXA' END                                                          AS prioridade_irrigacao,
       CASE WHEN dias_secos_consecutivos >= 10 THEN 'ALTO'
            WHEN dias_secos_consecutivos >= 5  THEN 'MÉDIO'
            ELSE 'BAIXO' END                                                          AS risco_estiagem,
       ROUND(area_plantada_ha * 10 * LEAST(chuva_dia_mm, 3.5), 1)                     AS agua_poupada_m3,
       ROUND(area_plantada_ha * 10 * LEAST(chuva_dia_mm, 3.5) * 1.85, 2)              AS economia_reais
FROM estiagem;

In [0]:
%sql

SELECT * FROM workspace.gs.ouro_mvp_diario ORDER BY municipio, data DESC LIMIT 20;

id_local,municipio,uf,mesorregiao,cultura_principal,area_plantada_ha,data,chuva_dia_mm,temp_media_c,temp_max_c,umidade_solo_media,horas_estresse_termico,dias_secos_consecutivos,saldo_hidrico_mm,prioridade_irrigacao,risco_estiagem,agua_poupada_m3,economia_reais
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-18,4.8,23.2,28.5,0.394,0,0,1.3,MODERADA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-17,0,22.8,28.2,0.427,0,1,-3.5,MODERADA,BAIXO,0,0
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-16,4.2,22.1,27.2,0.462,0,0,0.7,MODERADA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-15,15,22.4,28.9,0.374,0,0,11.5,BAIXA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-14,0.1,22.7,28.3,0.39,0,1,-3.4,MODERADA,BAIXO,14390.9,26623.17
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-13,9.2,22.2,28.4,0.453,0,0,5.7,BAIXA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-12,8.1,21.5,26.5,0.356,0,0,4.6,BAIXA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-11,1.5,21.7,26.9,0.364,0,0,-2,MODERADA,BAIXO,215863.5,399347.48
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-10,0.1,19.1,25.4,0.394,0,2,-3.4,MODERADA,BAIXO,14390.9,26623.17
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-09,0,17.2,21.1,0.439,0,1,-3.5,MODERADA,BAIXO,0,0


# AgroFácil · 04 · Cinco consultas SQL do MVP

Cada consulta parte de uma pergunta que o produto responde ao produtor rural.
Premissas comuns: necessidade hídrica de referência **3,5 mm/dia**; **1 mm sobre 1 ha = 10 m³**; água a **R$ 1,85/m³**;
dia seco = chuva do dia < 1 mm; estresse térmico = temperatura > 32 °C.

Cada consulta traz a pergunta, o SQL, o resultado e uma interpretação curta.

### Consulta 1 · Balanço hídrico
**Pergunta do MVP:** "Nos últimos dias choveu o suficiente para cobrir a necessidade de água da lavoura?"

In [0]:
%sql

WITH dia AS (
  SELECT l.municipio, t.data, ROUND(SUM(f.chuva_mm), 1) AS chuva_dia_mm
  FROM workspace.gs.ouro_fato_clima_lavoura f
  JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
  JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
  GROUP BY l.municipio, t.data
)
SELECT municipio, data, chuva_dia_mm,
       3.5 AS necessidade_mm,
       ROUND(chuva_dia_mm - 3.5, 1) AS saldo_dia_mm,
       ROUND(SUM(chuva_dia_mm - 3.5) OVER (PARTITION BY municipio ORDER BY data
                                           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS saldo_7_dias_mm
FROM dia
WHERE data >= DATE_SUB((SELECT MAX(data) FROM dia), 14)
ORDER BY municipio, data;

municipio,data,chuva_dia_mm,necessidade_mm,saldo_dia_mm,saldo_7_dias_mm
Campinas,2026-11-04,0.3,3.5,-3.2,-3.2
Campinas,2026-11-05,7.5,3.5,4,0.8
Campinas,2026-11-06,2.2,3.5,-1.3,-0.5
Campinas,2026-11-07,20,3.5,16.5,16
Campinas,2026-11-08,3,3.5,-0.5,15.5
Campinas,2026-11-09,0,3.5,-3.5,12
Campinas,2026-11-10,0.1,3.5,-3.4,8.6
Campinas,2026-11-11,1.5,3.5,-2,9.8
Campinas,2026-11-12,8.1,3.5,4.6,10.4
Campinas,2026-11-13,9.2,3.5,5.7,17.4


**Interpretação.** Em 18/11 o saldo hídrico acumulado de 7 dias está positivo nos três municípios (Campinas +16,9 mm, Piracicaba +8,4 mm e Ribeirão Preto +1,7 mm): a chuva da última semana cobriu a necessidade de 3,5 mm/dia. Ribeirão Preto é o mais próximo do limite, com saldo negativo de 14/11 a 17/11 (mínimo de −14,8 mm).

### Consulta 2 · Recomendação de irrigação
**Pergunta do MVP:** "Preciso irrigar hoje?" (último dia disponível, por município)

In [0]:
%sql

WITH dia AS (
  SELECT l.municipio, l.cultura_principal, t.data,
         ROUND(SUM(f.chuva_mm), 1)       AS chuva_dia_mm,
         ROUND(AVG(f.temperatura_c), 1)  AS temp_media_c,
         ROUND(AVG(f.umidade_solo), 3)   AS umidade_solo_media
  FROM workspace.gs.ouro_fato_clima_lavoura f
  JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
  JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
  GROUP BY l.municipio, l.cultura_principal, t.data
)
SELECT municipio, cultura_principal, data, chuva_dia_mm, temp_media_c, umidade_solo_media,
       CASE WHEN chuva_dia_mm < 2.0 AND (temp_media_c > 28.0 OR umidade_solo_media < 0.20) THEN 'ALTA - irrigar hoje'
            WHEN chuva_dia_mm < 5.0 THEN 'MODERADA - monitorar o solo'
            ELSE 'BAIXA - umidade adequada' END AS recomendacao
FROM dia
WHERE data = (SELECT MAX(data) FROM dia)
ORDER BY municipio;

municipio,cultura_principal,data,chuva_dia_mm,temp_media_c,umidade_solo_media,recomendacao
Campinas,Cana-de-açúcar,2026-11-18,4.8,23.2,0.394,MODERADA - monitorar o solo
Piracicaba,Cana-de-açúcar,2026-11-18,4.5,24.3,0.224,MODERADA - monitorar o solo
Ribeirão Preto,Cana-de-açúcar,2026-11-18,3.7,24.6,0.274,MODERADA - monitorar o solo


**Interpretação.** Em 18/11 os três municípios ficam em prioridade **MODERADA**: a chuva do dia (3,7 a 4,8 mm) ficou abaixo de 5 mm, mas nenhum atende ao critério de irrigação ALTA. Piracicaba merece mais atenção, pois sua umidade do solo (0,224 m³/m³) é a mais próxima do nível crítico de 0,20.

### Consulta 3 · Risco de estiagem
**Pergunta do MVP:** "Há risco de estiagem na minha região? Quantos dias seguidos sem chuva relevante?"

In [0]:
%sql

WITH dia AS (
  SELECT l.municipio, t.data, ROUND(SUM(f.chuva_mm), 1) AS chuva_dia_mm
  FROM workspace.gs.ouro_fato_clima_lavoura f
  JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
  JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
  GROUP BY l.municipio, t.data
),
marcado AS (
  SELECT *, CASE WHEN chuva_dia_mm < 1.0 THEN 1 ELSE 0 END AS dia_seco,
         DATEDIFF(data, DATE'2000-01-01')
           - ROW_NUMBER() OVER (PARTITION BY municipio, CASE WHEN chuva_dia_mm < 1.0 THEN 1 ELSE 0 END
                                ORDER BY data) AS grupo
  FROM dia
),
sequencias AS (
  SELECT municipio, grupo, MIN(data) AS inicio, MAX(data) AS fim, COUNT(*) AS dias_secos
  FROM marcado WHERE dia_seco = 1
  GROUP BY municipio, grupo
)
SELECT municipio,
       MAX(dias_secos) AS maior_sequencia_seca_dias,
       MAX_BY(inicio, dias_secos) AS inicio_da_maior,
       MAX_BY(fim, dias_secos) AS fim_da_maior,
       CASE WHEN MAX(dias_secos) >= 10 THEN 'ALTO' WHEN MAX(dias_secos) >= 5 THEN 'MÉDIO' ELSE 'BAIXO' END AS risco_estiagem
FROM sequencias
GROUP BY municipio
ORDER BY maior_sequencia_seca_dias DESC;

municipio,maior_sequencia_seca_dias,inicio_da_maior,fim_da_maior,risco_estiagem
Campinas,2,2026-10-26,2026-10-27,BAIXO
Piracicaba,2,2026-10-26,2026-10-27,BAIXO
Ribeirão Preto,2,2026-10-31,2026-11-01,BAIXO


**Interpretação.** A maior sequência de dias secos (chuva < 1 mm) foi de apenas 2 dias nos três municípios, então o risco de estiagem é **BAIXO** (o limite para MÉDIO é 5 dias seguidos). O período analisado, de 25/10 a 18/11, foi chuvoso.

### Consulta 4 · Estresse térmico
**Pergunta do MVP:** "Quantas horas de calor acima de 32 °C minha lavoura enfrentou?"

In [0]:
%sql

SELECT l.municipio, l.cultura_principal, t.ano, t.mes,
       COUNT(*)                         AS horas_estresse_termico,
       ROUND(MAX(f.temperatura_c), 1)   AS pico_temperatura_c,
       COUNT(DISTINCT t.data)           AS dias_com_estresse
FROM workspace.gs.ouro_fato_clima_lavoura f
JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
WHERE f.temperatura_c > 32.0
GROUP BY l.municipio, l.cultura_principal, t.ano, t.mes
ORDER BY horas_estresse_termico DESC;

municipio,cultura_principal,ano,mes,horas_estresse_termico,pico_temperatura_c,dias_com_estresse
Ribeirão Preto,Cana-de-açúcar,2026,11,20,35.4,4
Piracicaba,Cana-de-açúcar,2026,11,4,32.3,2


**Interpretação.** Ribeirão Preto concentra o estresse térmico: 20 horas acima de 32 °C em 4 dias, com pico de 35,4 °C. Piracicaba teve 4 horas em 2 dias (pico de 32,3 °C). Campinas não ultrapassou 32 °C e por isso não aparece no resultado.

### Consulta 5 · Impacto: água e dinheiro poupados
**Pergunta do MVP:** "Quanta água e quanto dinheiro deixei de gastar porque choveu?"

Água poupada no dia = área plantada (ha) x 10 m³/ha/mm x MIN(chuva do dia, 3,5 mm). Premissas: 10 m³/ha por mm (conversão física),
3,5 mm/dia (necessidade de referência) e R$ 1,85/m³ (custo da água, premissa do projeto).

In [0]:
%sql

WITH dia AS (
  SELECT l.municipio, l.area_plantada_total_ha AS area_ha, t.data, ROUND(SUM(f.chuva_mm), 1) AS chuva_dia_mm
  FROM workspace.gs.ouro_fato_clima_lavoura f
  JOIN workspace.gs.ouro_dim_local l ON l.id_local = f.id_local
  JOIN workspace.gs.ouro_dim_tempo t ON t.id_tempo = f.id_tempo
  GROUP BY l.municipio, l.area_plantada_total_ha, t.data
)
SELECT municipio,
       ROUND(area_ha, 0)                                                    AS area_plantada_ha,
       COUNT_IF(chuva_dia_mm >= 3.5)                                        AS dias_com_necessidade_coberta,
       ROUND(SUM(area_ha * 10 * LEAST(chuva_dia_mm, 3.5)), 0)               AS agua_poupada_m3,
       ROUND(SUM(area_ha * 10 * LEAST(chuva_dia_mm, 3.5)) * 1.85, 2)        AS economia_estimada_reais
FROM dia
GROUP BY municipio, area_ha
ORDER BY economia_estimada_reais DESC;

municipio,area_plantada_ha,dias_com_necessidade_coberta,agua_poupada_m3,economia_estimada_reais
Ribeirão Preto,63249,9,31118262,5.75687847E7
Piracicaba,47756,11,23495952,4.34675112E7
Campinas,14391,11,7454486,1.379079947E7


**Interpretação.** Estimativa teórica da água que a chuva substituiu e do custo evitado: Ribeirão Preto lidera (≈ 31,1 milhões de m³ e R$ 57,6 milhões) por ter a maior área plantada; Piracicaba soma ≈ R$ 43,5 milhões e Campinas ≈ R$ 13,8 milhões. São valores máximos, pois supõem que toda a área seria irrigada e que a chuva substitui a irrigação no limite de 3,5 mm/dia.

# AgroFácil · 05 · Governança LGPD

## Dados pessoais do pipeline e tratamento

| Dado | Onde aparece | Tratamento |
| :-- | :-- | :-- |
| Clima (Open-Meteo), cadastro de municípios (IBGE) e lavouras (simulado, sem pessoas) | Bronze, Prata, Ouro | **Não são dados pessoais**: fontes públicas, agregadas por município. Nenhuma ação necessária (minimização por desenho). |
| CPF do produtor | Cadastro simulado | **Hash SHA-256 com sal** (`id_produtor_hash`) + **máscara** (`***.456.789-**`). O CPF original nunca é gravado em tabela. |
| Nome do produtor | Cadastro simulado | **Máscara** (`Gi**** T***`). O nome original nunca é gravado em tabela. |
| E-mail, telefone, placa de veículo | Não coletados | **Não levar adiante** (minimização). |
| Localização exata de uma pessoa | Não coletada | O MVP usa só o município, nunca coordenadas de uma pessoa. |

**Base legal (simulação):** Art. 7º, V da LGPD (execução de contrato ou de procedimentos preliminares a pedido do titular),
pois o produtor contrata o serviço de recomendação. Não há coluna de consentimento: a base legal não é o consentimento.

**Dados simulados:** nomes e CPFs são fictícios. Os CPFs têm o dígito verificador **deliberadamente inválido**
(`gerador/gerar_produtores.py`), então nunca coincidem com o de uma pessoa real.

**Sal:** informado no widget `sal` em tempo de execução. **Nunca** escreva o sal no notebook, no código ou no repositório.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("sal", "", "Sal do hash (não versionar)")
SAL = dbutils.widgets.get("sal")
assert SAL, "Informe o sal no widget 'sal'. Ele não pode ficar no código nem no repositório."

produtores = [
    ('15881399811', 'Gisele Teixeira', '3538709', 'Laranja'),
    ('97309114116', 'Hugo Almeida', '3538709', 'Laranja'),
    ('53645568174', 'Fábio Barros', '3538709', 'Soja'),
    ('47294055700', 'Gisele Barros', '3543402', 'Laranja'),
    ('16897684619', 'Jorge Nogueira', '3538709', 'Cana-de-açúcar'),
    ('93208196717', 'Carla Nogueira', '3509502', 'Cana-de-açúcar'),
    ('77848648850', 'Hugo Machado', '3543402', 'Milho'),
    ('44002978710', 'Fábio Duarte', '3543402', 'Laranja'),
]
bruto = spark.createDataFrame(
    produtores, "cpf_produtor STRING, nome_produtor STRING, codigo_ibge STRING, cultura_principal STRING")

partes = F.split("nome_produtor", " ")
protegido = (
    bruto
    .withColumn("id_produtor_hash", F.sha2(F.concat(F.col("cpf_produtor"), F.lit(SAL)), 256))
    .withColumn("nome_mascarado",
                F.concat(F.substring(F.element_at(partes, 1), 1, 2), F.lit("**** "),
                         F.substring(F.element_at(partes, -1), 1, 1), F.lit("***")))
    .withColumn("cpf_mascarado",
                F.concat(F.lit("***."), F.substring("cpf_produtor", 4, 3), F.lit("."),
                         F.substring("cpf_produtor", 7, 3), F.lit("-**")))
    .drop("cpf_produtor", "nome_produtor")          # o dado original não segue adiante
)

municipios = spark.table("workspace.gs.prata_ibge_municipios").select("codigo_ibge", "municipio")
final = (protegido.join(municipios, "codigo_ibge", "left")
         .select("id_produtor_hash", "nome_mascarado", "cpf_mascarado", "codigo_ibge", "municipio",
                 "cultura_principal")
         .withColumn("base_legal_lgpd", F.lit("Art. 7º, V - execução de contrato (serviço de recomendação)"))
         .withColumn("finalidade", F.lit("Recomendação de manejo de irrigação ao produtor"))
         .withColumn("registrado_em", F.current_timestamp()))

final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gs.governanca_lgpd_produtores")

In [0]:
display(spark.table("workspace.gs.governanca_lgpd_produtores"))

id_produtor_hash,nome_mascarado,cpf_mascarado,codigo_ibge,municipio,cultura_principal,base_legal_lgpd,finalidade,registrado_em
b76920ec912e4e178a3a396f6911dbe9ba02d3563b94fc42c1407a1578a2348d,Jo**** N***,***.976.846-**,3538709,Piracicaba,Cana-de-açúcar,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
b58b865c7b81dc9b494b68c1485ce924af13c6b54fa7ec08072f2e32aa978c8a,Ca**** N***,***.081.967-**,3509502,Campinas,Cana-de-açúcar,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
1cf767dbb26420a160e369e506fcd728e009b053eadb608c5144c652c3156b9d,Fá**** D***,***.029.787-**,3543402,Ribeirão Preto,Laranja,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
1b66cb15f6e345c4ad7aebf4d6c7d321e27cefa8a5027a686d7089524a4d7692,Gi**** B***,***.940.557-**,3543402,Ribeirão Preto,Laranja,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
dcdd01d67d34d73c2407134d4b773e271e6a736d0aca9cbc694b3fac97827cd7,Hu**** M***,***.486.488-**,3543402,Ribeirão Preto,Milho,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
8ae31f179af29f678b1224316764bbce31c602f35a7dfeb82579b9e8b1cecbb9,Gi**** T***,***.813.998-**,3538709,Piracicaba,Laranja,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
13e27f6c6c47e51e68a908a0961096e479b7b9adcfc825f7a56344e5f424adeb,Hu**** A***,***.091.141-**,3538709,Piracicaba,Laranja,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z
c9f6316121b9fe19bd376da28719e81b436fec92c40d3e58f9ad80d3d7479c6d,Fá**** B***,***.455.681-**,3538709,Piracicaba,Soja,"Art. 7º, V - execução de contrato (serviço de recomendação)",Recomendação de manejo de irrigação ao produtor,2026-11-18T14:56:29.777Z


### Verificações
* Chaves `id_produtor_hash` únicas e não nulas.
* Nenhuma coluna com CPF ou nome em claro na tabela.

In [0]:
t = spark.table("workspace.gs.governanca_lgpd_produtores")
total = t.count()
distintos = t.select("id_produtor_hash").distinct().count()
nulos = t.filter(F.col("id_produtor_hash").isNull()).count()
colunas_proibidas = [c for c in t.columns if c in ("cpf_produtor", "nome_produtor", "cpf", "nome")]
print(f"linhas: {total} | hashes distintos: {distintos} | hashes nulos: {nulos}")
print(f"colunas com dado pessoal em claro: {colunas_proibidas or 'nenhuma'}")
assert total == distintos and nulos == 0 and not colunas_proibidas
print("OK: LGPD verificada")

linhas: 8 | hashes distintos: 8 | hashes nulos: 0
colunas com dado pessoal em claro: nenhuma
OK: LGPD verificada


# AgroFácil · 06 · Exportar a Ouro para o MVP

## Contrato de dados com o MVP

| Item | Definição |
| :-- | :-- |
| Objeto da Ouro consumido | view `workspace.gs.ouro_mvp_diario` (construída sobre `ouro_fato_clima_lavoura`, `ouro_dim_local` e `ouro_dim_tempo`) |
| Grão | 1 linha por município por dia |
| Frequência de atualização | diária (após o DAG coletar o dia e o pipeline rodar Bronze -> Prata -> Ouro) |
| Como o MVP lê | extrato JSON (`dados_ouro.json`) exportado desta view e publicado junto com o MVP (GitHub Pages) |
| Dados pessoais | nenhum (nível município). A tabela `governanca_lgpd_produtores` **não** é exposta ao MVP. |

| Coluna | Tipo | Descrição |
| :-- | :-- | :-- |
| `id_local` | string | código IBGE do município (7 dígitos) |
| `municipio`, `uf`, `mesorregiao` | string | contexto geográfico (cadastro IBGE) |
| `cultura_principal` | string | lavoura de maior área plantada (cadastro de lavouras simulado) |
| `area_plantada_ha` | double | área plantada total das lavouras do município (cadastro simulado) |
| `data` | date | dia (fuso America/Sao_Paulo) |
| `chuva_dia_mm`, `temp_media_c`, `temp_max_c`, `umidade_solo_media` | double | clima do dia (Open-Meteo) |
| `horas_estresse_termico` | int | horas com temperatura > 32 °C |
| `dias_secos_consecutivos` | int | dias seguidos com chuva < 1 mm |
| `saldo_hidrico_mm` | double | chuva do dia menos 3,5 mm de necessidade |
| `prioridade_irrigacao` | string | ALTA, MODERADA ou BAIXA |
| `risco_estiagem` | string | ALTO, MÉDIO ou BAIXO |
| `agua_poupada_m3`, `economia_reais` | double | impacto (premissas: 10 m³/ha/mm, 3,5 mm/dia, R$ 1,85/m³) |

Depois de rodar: baixe `dados_ouro.json` pela interface (Catalog > workspace > gs > chegada > `_export`) e salve em `mvp/dados_ouro.json` no repositório.

In [0]:
import json
import os

df = spark.table("workspace.gs.ouro_mvp_diario").orderBy("municipio", "data")
registros = [r.asDict(recursive=True) for r in df.collect()]  # poucas centenas de linhas
texto = json.dumps(registros, ensure_ascii=False, default=str, indent=1)

destino = "/Volumes/workspace/gs/chegada/_export/dados_ouro.json"
os.makedirs(os.path.dirname(destino), exist_ok=True)
with open(destino, "w", encoding="utf-8") as f:
    f.write(texto)

print(f"{len(registros)} linhas exportadas -> {destino} ({len(texto)/1024:.1f} KB)")
print(texto[:700])

75 linhas exportadas -> /Volumes/workspace/gs/chegada/_export/dados_ouro.json (38.0 KB)
[
 {
  "id_local": "3509502",
  "municipio": "Campinas",
  "uf": "SP",
  "mesorregiao": "Campinas",
  "cultura_principal": "Cana-de-açúcar",
  "area_plantada_ha": 14390.9,
  "data": "2026-10-25",
  "chuva_dia_mm": 43.7,
  "temp_media_c": 20.4,
  "temp_max_c": 23.6,
  "umidade_solo_media": 0.509,
  "horas_estresse_termico": 0,
  "dias_secos_consecutivos": 0,
  "saldo_hidrico_mm": 40.2,
  "prioridade_irrigacao": "BAIXA",
  "risco_estiagem": "BAIXO",
  "agua_poupada_m3": 503681.5,
  "economia_reais": 931810.78
 },
 {
  "id_local": "3509502",
  "municipio": "Campinas",
  "uf": "SP",
  "mesorregiao": "Campinas",
  "cultura_principal": "Cana-de-açúcar",
  "area_plantada_ha": 14390.9,
  "data": "20


In [0]:
display(df.limit(15))

id_local,municipio,uf,mesorregiao,cultura_principal,area_plantada_ha,data,chuva_dia_mm,temp_media_c,temp_max_c,umidade_solo_media,horas_estresse_termico,dias_secos_consecutivos,saldo_hidrico_mm,prioridade_irrigacao,risco_estiagem,agua_poupada_m3,economia_reais
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-25,43.7,20.4,23.6,0.509,0,0,40.2,BAIXA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-26,0.5,18.3,23,0.476,0,1,-3,MODERADA,BAIXO,71954.5,133115.83
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-27,0,17.9,24.4,0.428,0,2,-3.5,MODERADA,BAIXO,0,0
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-28,2.9,19.2,24.1,0.404,0,0,-0.6,MODERADA,BAIXO,417336.1,772071.79
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-29,17.4,19.3,21.7,0.467,0,0,13.9,BAIXA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-30,4.3,20.3,24,0.488,0,0,0.8,MODERADA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-10-31,0.6,21.8,26.9,0.44,0,1,-2.9,MODERADA,BAIXO,86345.4,159738.99
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-01,2.1,22.9,28.3,0.397,0,0,-1.4,MODERADA,BAIXO,302208.9,559086.47
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-02,11.6,21,24.2,0.447,0,0,8.1,BAIXA,BAIXO,503681.5,931810.78
3509502,Campinas,SP,Campinas,Cana-de-açúcar,14390.9,2026-11-03,0,23.4,29.3,0.414,0,1,-3.5,MODERADA,BAIXO,0,0
